In [1]:
import d3rlpy
from d3rlpy.dataset import InfiniteBuffer, ReplayBuffer
from d3rlpy.algos.transformer.decision_transformer import DTConstantRTGforFQE,DecisionTransformer
from d3rlpy.ope.fqe import FQE,FQEConfig
import time

on_server = False  # Set to True if running on the server
prefix = "/gpfs/data/fs72297/jklotz/programming/cloned_repos/forked_repos_for_master_thesis/" if on_server else "/home/julian/programming/cloned_repos/repos_for_master_thesis/"
cql_model = prefix + "d3rlpy_new/d3rlpy/experiments/exp02_qdt/d3rlpy_logs/CQL_Hopper-v4_1_20250710203053/model_400000.d3"
qdt_model = prefix + "d3rlpy_new/d3rlpy/experiments/exp02_qdt/d3rlpy_logs/QDT_hopper-medium-expert-v2_1_20250710151721/model_epoch_14.d3"

start_time = time.time()

dt_model = prefix + "d3rlpy/experiments/exp01_original_dt/slurm_files/d3rlpy_logs/gpu_array/DT_hopper-medium-expert-v2_1_20250710202727/model_epoch_12.d3"

device = "cuda:0" if on_server else "cpu"
dt_algo = d3rlpy.load_learnable(dt_model,device=device)
cql_algo = d3rlpy.load_learnable(cql_model,device=device)
# qdt_algo = d3rlpy.load_learnable(qdt_model,device=device)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/home/julian/miniconda3/envs/d3rlpy_dev_final_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from d3rlpy.metrics import evaluate_transformer_with_environment
import gymnasium as gym
def get_env_and_target_return(dataset):    
    if "halfcheetah" in dataset:
        return "HalfCheetah-v4", 6000
    elif "hopper" in dataset:
        return "Hopper-v4", 3800
    elif "walker" in dataset:
        return "Walker2d-v4", 5000
dataset= "hopper-medium-expert-v2"
env_name, target_return = get_env_and_target_return(dataset)
env = gym.make(env_name)

/home/julian/miniconda3/envs/d3rlpy_dev_final_py310/lib/python3.10/site-packages/gymnasium/envs/registration.py:517: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


In [3]:
#Evaluate the CQL model with the environment
from d3rlpy.dataset import ReplayBuffer
env_eval = d3rlpy.metrics.EnvironmentEvaluator(env, n_trials=10)
mean_return = env_eval(cql_algo,ReplayBuffer)

In [4]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

dataset = "hopper-medium-expert-v2"
if "halfcheetah" in dataset:
    env_name = "HalfCheetah-v4"
elif "hopper" in dataset:
    env_name = "Hopper-v4"
elif "walker" in dataset:
    env_name = "Walker2d-v4"

#env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v5"
print(env_name)

pkl_path = f"../../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{dataset}.pkl"

def convert_raw_episode(raw_ep):
    # Convert raw observations to a NumPy array and then to a list of individual observations.
    observations = np.array(raw_ep["observations"])

    # Ensure actions and rewards are NumPy arrays.
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    # For rewards, ensure they have an extra dimension (T, 1)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # Use the last element of "terminals" as the terminated flag.
    terminals = raw_ep["terminals"]
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)

Hopper-v4


In [5]:
from d3rlpy.dataset import ReplayBuffer, FIFOBuffer, BasicTransitionPicker
buffer_impl = FIFOBuffer(limit=10000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes,transition_picker=BasicTransitionPicker())
replay_buffer.sample_transition()

2025-08-22 09:30.21 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-08-22 09:30.21 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-08-22 09:30.21 [info     ] Action size has been automatically determined. action_size=3


Transition(observation=array([ 1.5030893 , -0.16527092, -0.52667576, -0.3751678 , -0.7149388 ,
        2.9886992 ,  0.751699  ,  0.8269754 ,  0.27716395,  1.3639189 ,
        3.4717014 ], dtype=float32), action=array([-0.45760483,  0.72649246, -0.00995274], dtype=float32), reward=array([4.016808], dtype=float32), next_observation=array([ 1.5087428 , -0.15933512, -0.52750146, -0.36018088, -0.68749195,
        3.0459507 ,  0.66085786,  0.65776634, -0.48224056,  2.382152  ,
        3.3900254 ], dtype=float32), next_action=array([-0.2474311 ,  0.40247446,  0.65083325], dtype=float32), terminal=0.0, interval=1, rewards_to_go=array([[4.016808 ],
       [4.065238 ],
       [4.1247764],
       [4.1473804],
       [4.163526 ],
       [4.134893 ],
       [4.1001883],
       [4.1134996],
       [4.0929856],
       [4.0723057],
       [4.0372915],
       [4.009066 ],
       [3.991824 ],
       [3.9407852],
       [3.8491194],
       [3.8138776],
       [3.7750862],
       [3.7566857],
       [3.78

In [6]:
from d3rlpy.dataset import ReplayBuffer, FIFOBuffer
buffer_impl = FIFOBuffer(limit=10000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)
replay_buffer.sample_transition()

2025-08-22 09:30.23 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-08-22 09:30.23 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-08-22 09:30.23 [info     ] Action size has been automatically determined. action_size=3


Transition(observation=array([  1.2780619 ,  -0.05973295,  -0.8883103 ,  -0.14174679,
         0.6076665 ,   2.8487136 ,  -1.254801  ,  -1.1985618 ,
         1.5260929 ,  -6.408379  , -10.        ], dtype=float32), action=array([ 0.810621 , -0.9542706, -0.9970264], dtype=float32), reward=array([3.82824], dtype=float32), next_observation=array([  1.2680972 ,  -0.06761004,  -0.8709979 ,  -0.19817227,
         0.51996666,   2.8096097 ,  -1.2390809 ,  -0.77247244,
         2.802756  ,  -7.693994  , -10.        ], dtype=float32), next_action=array([-0.05860196, -0.91448766, -0.9893821 ], dtype=float32), terminal=0.0, interval=1, rewards_to_go=array([[3.82824  ],
       [3.7293088],
       [3.6386566],
       [3.6577544],
       [3.606833 ],
       [3.576589 ],
       [3.6064813],
       [3.5940344],
       [3.6667736],
       [3.8005867],
       [3.9384294],
       [4.123591 ],
       [4.217097 ],
       [4.167097 ],
       [4.123726 ],
       [4.2551885],
       [4.5380664],
       [4.6491

In [7]:
from d3rlpy.dataset import ReplayBuffer, FIFOBuffer, EvalTrajectorySlicer

buffer_impl = FIFOBuffer(limit=10000000)
replay_buffer_dt_evaluation = ReplayBuffer(buffer=buffer_impl, episodes=episodes,trajectory_slicer=EvalTrajectorySlicer(1,3600))

2025-08-22 09:30.25 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-08-22 09:30.25 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-08-22 09:30.25 [info     ] Action size has been automatically determined. action_size=3


In [8]:
first_rtgs = []
for rtg in replay_buffer_dt_evaluation.sample_trajectory_batch(1000,20).returns_to_go:
    first_rtgs.append(rtg[0])
print(np.max(first_rtgs))

3600.0


In [9]:
first_rtgs = []
for rtg in replay_buffer.sample_trajectory_batch(10000,20).returns_to_go:
    first_rtgs.append(rtg[0])
print(len(first_rtgs))
print(np.max(first_rtgs))
print((np.array(first_rtgs)>3600).sum())

10000
3728.8467
43


In [10]:
first_rtgs = []

batch = replay_buffer.sample_trajectory_batch(10,20)
for rtg,rew in zip(batch.returns_to_go,batch.rewards):
    print(rtg[0:5]-rtg[1:6])
    print(rew[0:5])


[[3.8146973]
 [3.9077148]
 [3.9399414]
 [3.8947754]
 [3.751709 ]]
[[3.8146875]
 [3.907739 ]
 [3.93987  ]
 [3.894889 ]
 [3.7517178]]
[[2.7528076]
 [2.7426147]
 [2.7387695]
 [2.7271118]
 [2.6798706]]
[[2.7527966]
 [2.7426229]
 [2.7387767]
 [2.727127 ]
 [2.679853 ]]
[[3.5756836]
 [3.5549316]
 [3.5424805]
 [3.4875488]
 [3.4085693]]
[[3.5756946]
 [3.5549116]
 [3.5423858]
 [3.487605 ]
 [3.4085772]]
[[2.6488647]
 [2.5738525]
 [2.515564 ]
 [2.479309 ]
 [2.4855347]]
[[2.6488407]
 [2.5739243]
 [2.5155365]
 [2.4793189]
 [2.485522 ]]
[[3.8551025]
 [3.8536377]
 [3.8674316]
 [3.855713 ]
 [3.8444824]]
[[3.8551204]
 [3.853533 ]
 [3.8674862]
 [3.8557258]
 [3.8444862]]
[[3.4997559]
 [3.5898438]
 [3.6866455]
 [3.7634277]
 [3.7766113]]
[[3.4996657]
 [3.589892 ]
 [3.6867766]
 [3.7633724]
 [3.7764995]]
[[1.7316895]
 [1.7532959]
 [1.7657471]
 [1.8082275]
 [1.876831 ]]
[[1.7317002]
 [1.7532108]
 [1.7657739]
 [1.8082697]
 [1.8767263]]
[[4.3813477]
 [4.220581 ]
 [4.0733643]
 [3.9543457]
 [3.8963623]]
[[4.381376

In [11]:
replay_buffer_dt_evaluation.sample_trajectory_batch(64,20).rewards

array([[[0.        ],
        [0.        ],
        [0.        ],
        ...,
        [0.97612184],
        [0.98322695],
        [1.00354   ]],

       [[2.8766117 ],
        [2.8797817 ],
        [2.8806586 ],
        ...,
        [3.9819193 ],
        [4.1238775 ],
        [4.169634  ]],

       [[3.3155794 ],
        [3.3294356 ],
        [3.3255682 ],
        ...,
        [3.8641758 ],
        [4.0724764 ],
        [4.2418613 ]],

       ...,

       [[4.096087  ],
        [3.9523656 ],
        [3.8445947 ],
        ...,
        [3.4623852 ],
        [3.5547955 ],
        [3.6565363 ]],

       [[1.7674682 ],
        [1.7470019 ],
        [1.8336443 ],
        ...,
        [2.0212395 ],
        [2.049602  ],
        [2.097916  ]],

       [[3.1464076 ],
        [3.2164497 ],
        [3.2744215 ],
        ...,
        [3.4744983 ],
        [3.4605153 ],
        [3.4348304 ]]], dtype=float32)

In [12]:
from d3rlpy.ope.fqe import FQE,FQEConfig


fqe = FQE(algo=cql_algo,config=FQEConfig(),device=device)
fqe.fit(replay_buffer,n_steps=200,n_steps_per_epoch=100,batch_type="transition")

2025-08-22 09:30.28 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-08-22 09:30.28 [debug    ] Building models...            
2025-08-22 09:30.28 [debug    ] Models have been built.       
2025-08-22 09:30.28 [info     ] Directory is created at d3rlpy_logs/FQE_20250822093028
2025-08-22 09:30.28 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'fqe', 'params': {'batch_size': 100, 'gamma': 0.99, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'none', 'params': {}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': False, 'learning_rate': 0.0001, 'optim_factory': {'type': 'adam', 'params': 

Epoch 1/2: 100%|██████████| 100/100 [00:01<00:00, 97.05it/s, loss=3.77]

2025-08-22 09:30.29 [info     ] FQE_20250822093028: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.005863354206085205, 'time_algorithm_update': 0.004150919914245606, 'loss': 3.5112119019031525, 'time_step': 0.010179672241210937} step=100
2025-08-22 09:30.29 [info     ] Model parameters are saved to d3rlpy_logs/FQE_20250822093028/model_100.d3



Epoch 2/2: 100%|██████████| 100/100 [00:01<00:00, 96.89it/s, loss=2.6] 

2025-08-22 09:30.30 [info     ] FQE_20250822093028: epoch=2 step=200 epoch=2 metrics={'time_sample_batch': 0.006107316017150879, 'time_algorithm_update': 0.0039267754554748536, 'loss': 2.4211789965629578, 'time_step': 0.01019827127456665} step=200
2025-08-22 09:30.30 [info     ] Model parameters are saved to d3rlpy_logs/FQE_20250822093028/model_200.d3


[(1,
  {'time_sample_batch': 0.005863354206085205,
   'time_algorithm_update': 0.004150919914245606,
   'loss': 3.5112119019031525,
   'time_step': 0.010179672241210937}),
 (2,
  {'time_sample_batch': 0.006107316017150879,
   'time_algorithm_update': 0.0039267754554748536,
   'loss': 2.4211789965629578,
   'time_step': 0.01019827127456665})]

In [13]:
import numpy as np

init_obs = np.stack([ep.observations[0] for ep in replay_buffer.episodes])
print(init_obs.shape) 

actions = cql_algo.predict(init_obs)
init_val  = fqe.predict_value(init_obs, actions)    # (Nₑpisodes,)
print("Mean Vπ(s₀):", init_val.mean())

actions = cql_algo.sample_action(init_obs)
init_val  = fqe.predict_value(init_obs, actions)    # (Nₑpisodes,)
print("Mean Vπ(s₀) sample action:", init_val.mean())

(3213, 11)
Mean Vπ(s₀): 1.6675934
Mean Vπ(s₀) sample action: 1.6709105


In [14]:
from d3rlpy.ope.fqe import dt_predict_next_actions
from d3rlpy.torch_utility import TorchTrajectoryMiniBatch

In [15]:
trajectory_batch = replay_buffer.sample_trajectory_batch(64,20)

In [16]:
torch_batch = TorchTrajectoryMiniBatch.from_batch(batch=trajectory_batch,device="cpu")

In [17]:
pred, next_actions = dt_predict_next_actions(dt_algo._impl,traj=torch_batch)

In [18]:
print(len(trajectory_batch))
print(64*20)

64
1280


In [19]:
print(type(next_actions))
print(next_actions.shape)
print(next_actions.is_contiguous())

<class 'torch.Tensor'>
torch.Size([1216, 3])
True


In [20]:
transition_batch, _ = torch_batch.to_transition_batch()
print(type(transition_batch.next_actions))
print(transition_batch.next_actions.shape)
print(transition_batch.next_actions.is_contiguous())

<class 'torch.Tensor'>
torch.Size([1216, 3])
True


In [21]:
from d3rlpy.ope.fqe import FQETrajectory,FQEConfig

fqe = FQETrajectory(algo=dt_algo,config=FQEConfig(),device=device)
fqe.fit(replay_buffer,n_steps=200,n_steps_per_epoch=100)


2025-08-22 09:30.30 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-08-22 09:30.30 [debug    ] Building models...            
2025-08-22 09:30.30 [debug    ] Models have been built.       
2025-08-22 09:30.30 [info     ] Directory is created at d3rlpy_logs/FQETrajectory_20250822093030
2025-08-22 09:30.30 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'fqe', 'params': {'batch_size': 100, 'gamma': 0.99, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'none', 'params': {}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': False, 'learning_rate': 0.0001, 'optim_factory': {'type': 'adam', 

Epoch 1/2: 100%|██████████| 100/100 [00:16<00:00,  6.23it/s, loss=3.84]

2025-08-22 09:30.47 [info     ] FQETrajectory_20250822093030: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.010790514945983886, 'time_algorithm_update': 0.148389732837677, 'loss': 3.5835639595985413, 'time_step': 0.15935442209243775} step=100
2025-08-22 09:30.47 [info     ] Model parameters are saved to d3rlpy_logs/FQETrajectory_20250822093030/model_100.d3



Epoch 2/2: 100%|██████████| 100/100 [00:15<00:00,  6.27it/s, loss=2.45]

2025-08-22 09:31.03 [info     ] FQETrajectory_20250822093030: epoch=2 step=200 epoch=2 metrics={'time_sample_batch': 0.011339101791381836, 'time_algorithm_update': 0.1466951084136963, 'loss': 2.2866793805360794, 'time_step': 0.1581965923309326} step=200
2025-08-22 09:31.03 [info     ] Model parameters are saved to d3rlpy_logs/FQETrajectory_20250822093030/model_200.d3


[(1,
  {'time_sample_batch': 0.010790514945983886,
   'time_algorithm_update': 0.148389732837677,
   'loss': 3.5835639595985413,
   'time_step': 0.15935442209243775}),
 (2,
  {'time_sample_batch': 0.011339101791381836,
   'time_algorithm_update': 0.1466951084136963,
   'loss': 2.2866793805360794,
   'time_step': 0.1581965923309326})]

In [22]:
start_time = time.time()
for i in range(10000):
    dt_algo.as_stateful_wrapper(2000)
print(time.time()-start_time)

0.03358292579650879


In [ ]:
# def evaluate_initial_obs_transformer(
#         replay_buffer,
#         dt_algo,
#     ):
#     for episode in replay_buffer.sample_trajectory():
#         episodes

In [25]:
import numpy as np

init_obs = np.stack([ep.observations[0] for ep in replay_buffer.episodes])
print(init_obs.shape) 

actions = cql_algo.predict(init_obs)
init_val  = fqe.predict_value(init_obs, actions)    # (Nₑpisodes,)
print("Mean Vπ(s₀):", init_val.mean())

actions = cql_algo.sample_action(init_obs)
init_val  = fqe.predict_value(init_obs, actions)    # (Nₑpisodes,)
print("Mean Vπ(s₀) sample action:", init_val.mean())

(3213, 11)
Mean Vπ(s₀): 1.5453997
Mean Vπ(s₀) sample action: 1.5527858


In [16]:
trajectory_batch = replay_buffer.sample_trajectory_batch(64,20)

In [17]:
type(trajectory_batch)

d3rlpy.dataset.mini_batch.TrajectoryMiniBatch

In [18]:
trajectory_batch.returns_to_go

array([[[ 990.41187],
        [ 988.0391 ],
        [ 985.69354],
        ...,
        [ 940.6619 ],
        [ 937.0776 ],
        [ 933.50964]],

       [[   0.     ],
        [   0.     ],
        [   0.     ],
        ...,
        [1440.3129 ],
        [1438.9512 ],
        [1437.5399 ]],

       [[1967.4893 ],
        [1963.7902 ],
        [1959.7853 ],
        ...,
        [1898.1396 ],
        [1894.6006 ],
        [1891.2522 ]],

       ...,

       [[1224.5249 ],
        [1220.6063 ],
        [1216.6603 ],
        ...,
        [1155.1094 ],
        [1151.2021 ],
        [1147.3854 ]],

       [[ 937.2705 ],
        [ 934.5731 ],
        [ 931.8888 ],
        ...,
        [ 893.15076],
        [ 890.64343],
        [ 888.11224]],

       [[ 882.4002 ],
        [ 879.2868 ],
        [ 876.17755],
        ...,
        [ 830.5336 ],
        [ 827.4123 ],
        [ 824.2667 ]]], dtype=float32)

In [19]:
replay_buffer.sample_transition().rewards_to_go

array([[3.50818  ],
       [3.4645615],
       [3.3700023],
       [3.3286376],
       [3.3785746],
       [3.3625906],
       [3.3395584],
       [3.348785 ],
       [3.370838 ],
       [3.3718185],
       [3.377689 ],
       [3.4793167],
       [3.51999  ],
       [3.522614 ],
       [3.5509276],
       [3.533806 ],
       [3.536581 ],
       [3.514646 ],
       [3.5298944],
       [3.533988 ],
       [3.5314415],
       [3.5276525],
       [3.5427113],
       [3.5922458],
       [3.460211 ],
       [3.4186702],
       [3.8124278],
       [4.230324 ],
       [4.363751 ],
       [4.4404817],
       [4.470203 ],
       [4.4841847],
       [4.5198717],
       [4.5182543],
       [4.4280972],
       [4.198467 ],
       [4.0222025],
       [3.9349556],
       [3.9338799],
       [3.930072 ],
       [3.882888 ],
       [3.8330295],
       [3.7786703],
       [3.7089858],
       [3.6113741],
       [3.4787917],
       [3.3453345],
       [3.2448523],
       [3.2205071],
       [3.256986 ],


In [20]:
gamma = 0.9
rewards = [1,2,3,5,6]
gammas = gamma ** np.arange(len(rewards)-1, -1, -1)
gammas = gammas.reshape(-1, 1).astype(np.float32)
print(gammas)
discounted_sum = np.sum(gammas * rewards, axis=0)  # shape (1,)
print(discounted_sum)

[[0.6561]
 [0.729 ]
 [0.81  ]
 [0.9   ]
 [1.    ]]
[ 4.09509993  8.19019985 12.28529978 20.47549963 24.57059956]


In [21]:
import numpy as np

gamma = 0.9
rewards = np.array([1, 2, 3, 5, 6], dtype=np.float32)

# Reverse: reward_0 has highest power
gammas = gamma ** np.arange(len(rewards)-1, -1, -1)  # shape (5,)

# Correct element-wise multiplication
discounted_sum = np.sum(gammas * rewards)  # scalar
print("Gammas:", gammas)
print("Discounted sum:", discounted_sum)


Gammas: [0.6561 0.729  0.81   0.9    1.    ]
Discounted sum: 15.0441


In [22]:
import numpy as np

gamma = 0.9
rewards = np.array([1, 2, 3], dtype=np.float32)

# Reverse: reward_0 has highest power
gammas = gamma ** np.arange(len(rewards)-1, -1, -1)  # shape (5,)

# Correct element-wise multiplication
discounted_sum = np.sum(gammas * rewards)  # scalar
print("gamma:",gamma)
print("rewards",rewards)
print("Gammas:", gammas)
print("Discounted sum:", discounted_sum)

import numpy as np

gamma = 0.9
rewards = np.array([1,2], dtype=np.float32)

# Reverse: reward_0 has highest power
gammas = gamma ** np.arange(len(rewards)-1, -1, -1)  # shape (5,)

# Correct element-wise multiplication
discounted_sum = np.sum(gammas * rewards)  # scalar
print("gamma:",gamma)
print("rewards",rewards)
print("Gammas:", gammas)
print("Discounted sum:", discounted_sum)

import numpy as np

gamma = 0.9
rewards = np.array([1], dtype=np.float32)

# Reverse: reward_0 has highest power
gammas = gamma ** np.arange(len(rewards)-1, -1, -1)  # shape (5,)

# Correct element-wise multiplication
discounted_sum = np.sum(gammas * rewards)  # scalar
print("gamma:",gamma)
print("rewards",rewards)
print("Gammas:", gammas)
print("Discounted sum:", discounted_sum)

gamma: 0.9
rewards [1. 2. 3.]
Gammas: [0.81 0.9  1.  ]
Discounted sum: 5.61
gamma: 0.9
rewards [1. 2.]
Gammas: [0.9 1. ]
Discounted sum: 2.9
gamma: 0.9
rewards [1.]
Gammas: [1.]
Discounted sum: 1.0
